# Build the v57 figure reports

Assembles, from what the figure notebooks wrote:
* `manuscript/figures/README.md` — figure / panel → notebook → source files (sha256) → task ID, from
  `manuscript/figures/provenance.json`;
* `manuscript/FIGURE_LEGENDS_v57.md` — from `manuscript/figures/legends/*.md` (numbers formatted by each
  figure notebook from the source files);
* the Supplementary Tables S2/S3 section of the README (tables written by `notebooks/tables_v57.ipynb`);
* `manuscript/REPRODUCE_v57.md` — static instructions; the runtime block is filled in by `run_all.py`.

In [1]:
from _style import *
import json as _json

prov = _json.loads(PROV_PATH.read_text())
MAIN = [f"Fig{i}" for i in range(1, 8)]
SUPP = sorted((k for k in prov if k.startswith("FigS")), key=lambda k: int(k[4:]))
assert SUPP == [f"FigS{i}" for i in range(1, 5)], SUPP
assert not [f for f in prov if f.startswith("Fig") and f not in MAIN + SUPP], "unexpected figures in provenance.json"
VAL = {f: _json.loads((VALUES_DIR / f"{f}.json").read_text()) for f in MAIN + SUPP}
from PIL import Image
def png_px(f):
    return Image.open(REPO / f"manuscript/figures/{prov[f]['kind']}/{f}.png").size
missing = [f for f in MAIN + SUPP if f not in prov]
assert not missing, f"figures missing from provenance.json: {missing}"
for f in MAIN + SUPP:
    for o in prov[f]["outputs"]:
        assert (REPO / o).exists(), o

In [2]:
def pend_list(entry):
    return [p for p, d in entry["panels"].items() if d["status"] == "placeholder"]

L = ["# v57 figures — provenance", "",
     "Generated by `notebooks/build_reports.ipynb` from `provenance.json` (written by each figure notebook at run time; do not edit by hand).",
     "Every plotted or quoted value is computed in the named notebook from the listed files. sha256 values are those of the files read in the last run.",
     "Regenerate everything with `.venv-fig/bin/python notebooks/run_all.py` (see `manuscript/REPRODUCE_v57.md`).", "",
     "**Formats:** every figure is written as vector PDF (TrueType fonts embedded, `pdf.fonttype = 42`), PNG at 600 dpi and TIFF at 600 dpi (LZW, RGB). "
     "All figures are drawn at journal width (double column 183 mm; minimum font 6 pt and minimum line width 0.5 pt enforced at save time in `notebooks/_style.py`). "
     "The scJDO v1 (v56) figures are not part of v57; they remain in version 56 of the preprint. Visual QA: `manuscript/figures/QA_v57.md`.", "",
     "**S15** is complete (all 130 analysis checkpoints, zero pending units) and merged; Fig3d, Fig7 and the S15 rows of NUMBERS_v57 read "
     "`benchmarks/synthetic_multimodal/s15/` (`S15_DIR`). Fig3e shows the S14b and S14c test-construction results. "
     "Fig6 panel d shows the merged S22 (`benchmarks/real_flow/s22/`) and S18d (`benchmarks/perturb_dose/`) results. "
     "Supplementary Tables S2 (S22) and S3 (S18d) are in `manuscript/tables/` (see the section below).", "",
     "## Figure files", "", "| figure | kind | PDF / PNG / TIFF | PNG pixels (600 dpi) | print size (mm) |", "|---|---|---|---|---|"]
for f in MAIN + SUPP:
    e = prov[f]; w, h = png_px(f)
    L.append(f"| {f} | {e['kind']} | `{e['kind']}/{f}.{{pdf,png,tiff}}` | {w} × {h} | {w / 600 * 25.4:.1f} × {h / 600 * 25.4:.1f} |")
L.append("")
for kind, figs in [("Main figures", MAIN), ("Supplementary figures", SUPP)]:
    L += [f"## {kind}", ""]
    for f in figs:
        e = prov[f]
        L += [f"### {f}{' (INTERIM)' if e['interim'] else ''}", "",
              f"- Notebook: `{e['notebook']}`", f"- Outputs: {', '.join('`' + o + '`' for o in e['outputs'])}"]
        if pend_list(e):
            L.append(f"- Placeholder panels: {', '.join(pend_list(e))}")
        L += ["", "| panel | content | status |", "|---|---|---|"]
        for p, d in e["panels"].items():
            L.append(f"| {p} | {d['description']} | {d['status']} |")
        L += ["", "| source file | sha256 | panels | task | note |", "|---|---|---|---|---|"]
        for s, d in e["sources"].items():
            L.append(f"| `{s}` | `{d['sha256']}` | {', '.join(d['panels'])} | {', '.join(d['tasks'])} | {'; '.join(d['notes'])} |")
        L.append("")
if "TABLES" in prov:
    e = prov["TABLES"]
    L += ["## Supplementary tables", "", f"- Notebook: `{e['notebook']}` → `manuscript/tables/TableS2_S22.{{csv,md}}`, `manuscript/tables/TableS3_S18d.{{csv,md}}` "
          "(each .md carries the table caption: what, n, pre-registered vs report only)", "",
          "| table | content | status |", "|---|---|---|"]
    for p, d in e["panels"].items():
        L.append(f"| {p} | {d['description']} | {d['status']} |")
    L += ["", "| source file | sha256 | tables | task |", "|---|---|---|---|"]
    for s, d in e["sources"].items():
        L.append(f"| `{s}` | `{d['sha256']}` | {', '.join(d['panels'])} | {', '.join(d['tasks'])} |")
    L.append("")
if "NUMBERS" in prov:
    e = prov["NUMBERS"]
    L += ["## Number provenance report", "", f"- Notebook: `{e['notebook']}` → `manuscript/NUMBERS_v57.md`, `manuscript/figures/numbers_v57.csv`",
          f"- Reads {len(e['sources'])} source files (listed with sha256 at the end of NUMBERS_v57.md).", ""]
(FIGDIR / "README.md").write_text("\n".join(L) + "\n")

24541

In [3]:
G = ["# Figure legends — manuscript v57", "",
     "Generated by `notebooks/build_reports.ipynb` from the legend text written by each figure notebook; every number is formatted from the source "
     "files at run time (see `manuscript/figures/README.md` for files and sha256). Labels: *pre-registered* = gate fixed in the task before execution; "
     "*report only* = fixed in the task but not gated; *post hoc* = chosen after results.", "",
     "Figure plan: Fig1 framework, Fig2 benchmark, Fig3 flow operator, Fig4 dyngen, Fig5 real data and scale, Fig6 score operator, "
     "Fig7 archetypes; Supplementary Figures S1–S4. scJDO v1 (v56) figures are not carried over.", "", "## Main figures", ""]
for f in MAIN:
    G += [(LEGEND_DIR / f"{f}.md").read_text().strip(), ""]
G += ["## Supplementary figures", ""]
for f in SUPP:
    G += [(LEGEND_DIR / f"{f}.md").read_text().strip(), ""]
(REPO / "manuscript" / "FIGURE_LEGENDS_v57.md").write_text("\n".join(G) + "\n")

18463

In [4]:
import shutil as _sh, subprocess as _sp
_t = _sh.which("pdftoppm")
TOOL = ((_sp.run([_t, "-v"], capture_output=True, text=True).stderr or "").splitlines() or ["pdftoppm"])[0].strip() if _t else "pdftoppm"
R = f"""# Reproducing the v57 figures and number reports

## Environment (private to this worktree)

```bash
cd <repo root>                      # branch figures-v57
python3 -m venv .venv-fig           # Python 3.13 was used
.venv-fig/bin/pip install matplotlib pandas numpy scipy nbformat nbclient ipykernel
```

`.venv-fig/` is git-ignored. Nothing is registered globally: `run_all.py` puts the venv's `bin/` first on `PATH`,
so nbclient starts the venv's own `python3` kernel.

## Run

```bash
.venv-fig/bin/python notebooks/run_all.py              # all notebooks, in order, executed in place
.venv-fig/bin/python notebooks/run_all.py fig3_flow_operator   # a subset (does not reset provenance.json)
```

A full run deletes and rebuilds `manuscript/figures/provenance.json`, executes (per-cell timeout 600 s, first error aborts with a
non-zero exit and leaves the failing notebook's outputs in place):

{chr(10).join(f'{i + 1}. `notebooks/{n}.ipynb`' for i, n in enumerate(['fig1_framework', 'fig2_benchmark', 'fig3_flow_operator', 'fig4_dyngen', 'fig5_real_scale', 'fig6_score_operator', 'fig7_archetypes', 'supp_s10_diffusion', 'supp_s21_three_timepoints', 'supp_s13_whitening', 'supp_s11_operator_agreement', 'tables_v57', 'numbers_v57', 'build_reports']))}

and writes `manuscript/figures/run_log.json` (wall time per notebook).

Outputs: `manuscript/figures/main/Fig1–7.{{pdf,png,tiff}}`, `manuscript/figures/supplementary/FigS1–S4.{{pdf,png,tiff}}`,
`manuscript/tables/TableS2_S22.{{csv,md}}` and `manuscript/tables/TableS3_S18d.{{csv,md}}` (Supplementary Tables S2, S3),
`manuscript/figures/provenance.json`, `manuscript/figures/values/*.json` (every number used in a legend), `manuscript/figures/README.md`,
`manuscript/FIGURE_LEGENDS_v57.md`, `manuscript/NUMBERS_v57.md`, `manuscript/figures/numbers_v57.csv`, this file.
Shared style and provenance helpers: `notebooks/_style.py`.

## Formats and resolution

- Set centrally in `notebooks/_style.py` (`DPI = 600`, `FORMATS = ("pdf", "png", "tiff")`, `save()`): vector PDF with TrueType-embedded fonts
  (`pdf.fonttype = 42`), PNG at 600 dpi, TIFF at 600 dpi (LZW compression, RGB). Widths: single column 89 mm, 1.5 column 120 mm,
  double column 183 mm, maximum height 240 mm (`save()` prints the final size and warns above 183 × 240 mm).
- At save time every text is clamped to ≥ 6 pt and every non-zero line width to ≥ 0.5 pt; bold panel letters are placed top-left, outside the
  axes, tick labels and title of their panel and aligned per row. Point labels that could collide use `repel_labels()`.

## Inputs

All inputs of the v57 figures (Fig1–7, Supplementary Figs S1–S4), of Supplementary Tables S2/S3 and of NUMBERS_v57 are committed result files under `benchmarks/` (S15 at
`benchmarks/synthetic_multimodal/s15/`, the single variable `S15_DIR` in `notebooks/_style.py`, overridable with `SCJDO2_S15_DIR`).
The scJDO v1 (v56) figures are not part of v57 (they remain in version 56 of the preprint); no v56 file is read.

## Runtime

<!-- runtime:start -->
(filled in by run_all.py after a full run)
<!-- runtime:end -->

## Pending (regenerate after each item lands)

- S22 (`benchmarks/real_flow/s22/`) and S18d (`benchmarks/perturb_dose/`) are merged: Fig6d, Supplementary Tables S2/S3 and their
  NUMBERS_v57 rows are computed from them (no longer pending).
- **S14c** (confirmatory transition test) ended as a construction failure (2 of 4 required seeds eligible; no model fitted); Fig3e shows it.
- **S16** (head-to-head with CellRank / spliceJAC / dynamo): no figure (run but not scored; not reported in the manuscript).
- dyngen s18 inversion diagnosis: pending (noted in the Fig4 legend).
"""
(REPO / "manuscript" / "REPRODUCE_v57.md").write_text(R)
print("reports written")

reports written
